# Kaggle — LightGBM, XGBoost, Random Forest

Upload the **same ZIP containing one train CSV and one test CSV** that the Vast notebook will use. Set `DATA_ZIP` in the config when Kaggle input has multiple ZIPs. Validation is 20% of train; test stays held out. The tree models use 5-fold CV on the original train file, then final models train on the 80% fit partition and tune thresholds on the 20% validation partition.
**Compute:** XGBoost trains on CUDA and fails the preflight if CUDA is unavailable. The scikit-learn Random Forest uses CPU. LightGBM uses CPU by default; set `LGB_DEVICE="gpu"` only after confirming OpenCL works in your Kaggle session. The whole train and test files are loaded; there is no sample cap. A GPU accelerator does not guarantee this full five-fold run will fit Kaggle’s session time or RAM.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','lightgbm','xgboost',
                'scikit-learn>=1.4','pandas','numpy','scipy','joblib','matplotlib'],check=True)
print('Packages installed. If this is the first installation, restart the kernel before continuing.')

In [ ]:
import json, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import scipy.sparse as sp
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,
                             precision_recall_curve,classification_report,confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

INPUT_ROOT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working/ml_results') if Path('/kaggle/working').exists() else Path('./ml_results')
OUT_DIR.mkdir(parents=True,exist_ok=True)
N_FOLDS = 5
LGB_DEVICE = "cpu"  # Optional "gpu" needs a working OpenCL device and LightGBM GPU build.
TARGET = 'Severity'
RANDOM_STATE = 42
VAL_SIZE = 0.20
MAX_CAT_ONEHOT = 50
USE_CLASS_WEIGHT = True
CLASS_WEIGHT_CAP = 2.5
DATETIME_COLS = ['Start_Time', 'End_Time', 'Weather_Timestamp']
BOOL_COLS = ['Amenity','Crossing','Give_Way','Junction','No_Exit','Railway','Station','Stop','Traffic_Signal']
GEO_COLS = ['Start_Lat', 'Start_Lng']
YELLOW_EXCLUDE = ['ID','Source','End_Lat','End_Lng','Country','Wind_Chill(F)',
                  'Precipitation(in)','Bump','Roundabout','Traffic_Calming','Turning_Loop']
METRICS = ['Accuracy','Macro_Precision','Macro_Recall','Macro_F1']

# Set this explicitly if your input location has more than one ZIP.
DATA_ZIP = "/workspace/upload/accidents.zip"
TRAIN_CSV = None
TEST_CSV = None

In [ ]:
import subprocess

def check_compute():
    try:
        result = subprocess.run(['nvidia-smi', '-L'], check=True, capture_output=True, text=True)
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError('Select a GPU accelerator in Kaggle Notebook settings before running.') from exc
    print('Detected GPU:', result.stdout.strip())
    assert xgb.build_info().get('USE_CUDA', False), 'This XGBoost build has no CUDA support.'
    probe = xgb.XGBClassifier(n_estimators=2, max_depth=2, tree_method='hist',
                              device='cuda', objective='multi:softprob', num_class=3,
                              eval_metric='mlogloss', random_state=RANDOM_STATE)
    probe.fit(np.arange(90, dtype=np.float32).reshape(30, 3),
              np.tile(np.arange(3), 10))
    actual = json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
    assert actual.startswith('cuda'), f'XGBoost fell back to {actual}; fix GPU setup before full training.'
    print('XGBoost CUDA preflight passed:', actual)
    if LGB_DEVICE == 'gpu':
        sample = np.arange(120, dtype=np.float32).reshape(40, 3)
        try:
            lgb.LGBMClassifier(n_estimators=2, min_child_samples=1,
                               device_type='gpu', verbosity=-1).fit(sample,
                                   np.tile(np.arange(4), 10))
        except Exception as exc:
            raise RuntimeError('LightGBM OpenCL GPU probe failed; use LGB_DEVICE="cpu" or install a working OpenCL runtime.') from exc
        print('LightGBM OpenCL GPU preflight passed')
    print('RandomForestClassifier: CPU training via n_jobs=-1')

check_compute()


## Load the existing train/test ZIP and create validation

In [ ]:
def engineer_features(df):
    df = df.copy()
    df = df.drop(columns=YELLOW_EXCLUDE, errors="ignore")
    # End_Time occurs after an accident and must not define an input feature.
    df = df.drop(columns=["End_Time"], errors="ignore")
    for c in DATETIME_COLS:
        if c in df.columns: df[c] = pd.to_datetime(df[c], errors="coerce")
    if "Start_Time" in df.columns:
        df["Start_Hour"] = df["Start_Time"].dt.hour
        df["Start_Dayofweek"] = df["Start_Time"].dt.dayofweek
        df["Start_IsWeekend"] = (df["Start_Dayofweek"] >= 5).astype(int)
        df["Is_RushHour"] = df["Start_Hour"].isin([7,8,9,16,17,18]).astype(int)
    if "Weather_Timestamp" in df.columns:
        df["Weather_Hour"] = df["Weather_Timestamp"].dt.hour
    df = df.drop(columns=[c for c in DATETIME_COLS if c in df.columns], errors="ignore")
    for c in BOOL_COLS:
        if c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].map({"True":1,"False":0,True:1,False:0,"TRUE":1,"FALSE":0,"Y":1,"N":0})
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    if "Weather_Condition" in df.columns:
        df["Weather_Condition"] = (df["Weather_Condition"].astype(str).str.lower().str.strip()
                                   .str.replace(r"[^a-z ]","",regex=True))
        # Frequency/target encoding learns its vocabulary from each training fold only.
    for c in df.columns:
        if df[c].dtype == object: df[c] = df[c].fillna("Missing").astype(str)
    df = df.dropna(subset=[TARGET])  # keep rows with missing coordinates
    return df

In [ ]:
import hashlib
from zipfile import ZipFile

def locate_inputs(search_root):
    if DATA_ZIP is not None:
        archives = [Path(DATA_ZIP)]
    else:
        archives = list(Path(search_root).rglob('*.zip'))
    if archives:
        if len(archives) != 1:
            raise ValueError(f'Found {len(archives)} ZIPs; set DATA_ZIP to the intended ZIP: {archives}')
        archive = archives[0]
        with ZipFile(archive) as z:
            csvs = [s for s in z.namelist() if s.lower().endswith('.csv') and not s.startswith('__MACOSX/')]
            train = [s for s in csvs if 'train' in Path(s).name.lower()]
            test = [s for s in csvs if 'test' in Path(s).name.lower()]
            if len(train) != 1 or len(test) != 1:
                raise ValueError(f'Expected one train CSV and one test CSV in {archive}; found: {csvs}')
            with z.open(train[0]) as f: train_df = pd.read_csv(f, low_memory=False)
            with z.open(test[0]) as f: test_df = pd.read_csv(f, low_memory=False)
            info_tr, info_te = z.getinfo(train[0]), z.getinfo(test[0])
            identity = {'zip':str(archive), 'train_member':train[0], 'test_member':test[0],
                        'train_crc32':f'{info_tr.CRC:08x}', 'test_crc32':f'{info_te.CRC:08x}',
                        'train_bytes':info_tr.file_size, 'test_bytes':info_te.file_size}
        print('Loaded ZIP:', archive, '| train:', train[0], '| test:', test[0])
        return train_df, test_df, identity
    if TRAIN_CSV is None and TEST_CSV is None:
        csvs=list(Path(search_root).rglob('*.csv'))
        trains=[p for p in csvs if 'train' in p.name.lower()]
        tests=[p for p in csvs if 'test' in p.name.lower()]
        if len(trains)!=1 or len(tests)!=1:
            raise FileNotFoundError(f'Expected one train CSV and one test CSV under {search_root}; set TRAIN_CSV and TEST_CSV explicitly. Found: {csvs}')
        train_path,test_path=trains[0],tests[0]
    elif TRAIN_CSV is not None and TEST_CSV is not None:
        train_path,test_path=Path(TRAIN_CSV),Path(TEST_CSV)
    else:
        raise ValueError('Set both TRAIN_CSV and TEST_CSV, or neither.')
    if not train_path.is_file() or not test_path.is_file():
        raise FileNotFoundError((train_path, test_path))
    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(8*1024*1024), b''): h.update(block)
        return h.hexdigest()
    return (pd.read_csv(train_path,low_memory=False),pd.read_csv(test_path,low_memory=False),
            {'train_sha256':digest(train_path),'test_sha256':digest(test_path)})

df_train, df_test, input_identity = locate_inputs(INPUT_ROOT)
assert TARGET in df_train.columns and TARGET in df_test.columns, 'Both files need Severity labels.'
df_train = engineer_features(df_train)
df_test = engineer_features(df_test)
classes = sorted(df_train[TARGET].astype(int).unique())
class_map = {int(c):i for i,c in enumerate(classes)}
y_dev = df_train[TARGET].astype(int).map(class_map).to_numpy()
y_test = df_test[TARGET].astype(int).map(class_map).to_numpy()
assert np.isfinite(y_test).all(), 'Test file has a Severity class absent from train.'
y_test = y_test.astype(int)
n_classes = len(classes)
X_dev = df_train.drop(columns=[TARGET])
X_test = df_test.drop(columns=[TARGET])
assert set(X_dev.columns) == set(X_test.columns), 'Train/test feature columns differ.'
X_test = X_test[X_dev.columns]
numeric_cols = [c for c in X_dev if pd.api.types.is_numeric_dtype(X_dev[c])]
cat_cols = [c for c in X_dev if c not in numeric_cols]
high_card = [c for c in cat_cols if X_dev[c].nunique() > MAX_CAT_ONEHOT]
low_card = [c for c in cat_cols if c not in high_card]
X_fit, X_val, y_fit, y_val = train_test_split(
    X_dev, y_dev, test_size=VAL_SIZE, stratify=y_dev, random_state=RANDOM_STATE)
manifest = dict(input_identity, train_rows=len(X_dev), validation_rows=len(X_val),
    final_fit_rows=len(X_fit), test_rows=len(X_test), class_map=class_map,
    validation_index_sha256=hashlib.sha256(np.sort(X_val.index.to_numpy(dtype=np.int64)).tobytes()).hexdigest())
(OUT_DIR / 'split_manifest.json').write_text(json.dumps(manifest,indent=2))
joblib.dump(class_map, OUT_DIR / 'class_map.pkl')
print('Train/validation/test:', len(X_fit),len(X_val),len(X_test), '| classes:', class_map)
print('Missing coordinate rows retained:', int(X_dev[GEO_COLS].isna().any(axis=1).sum()))

## Shared preprocessing and metrics

In [ ]:
def calc_metrics(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Macro_Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro_Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0)}


THRESHOLD_BETA = 1.0

def tune_thresholds(y_val, proba_val, classes, beta=THRESHOLD_BETA):
    """Per-class decision thresholds maximizing F_beta on validation probabilities.
    Shared by LightGBM / XGBoost / RF / MLP / ST-GNN."""
    y_val = np.asarray(y_val); thresholds = {}
    for i, c in enumerate(classes):
        y_bin = (y_val == c).astype(int)
        prec, rec, thr = precision_recall_curve(y_bin, proba_val[:, i])
        thr = np.append(thr, 1.0)
        with np.errstate(divide="ignore", invalid="ignore"):
            fbeta = (1 + beta**2) * prec * rec / (beta**2 * prec + rec)
        fbeta = np.nan_to_num(fbeta)
        thresholds[c] = float(thr[np.argmax(fbeta)])
    return thresholds

def predict_with_thresholds(proba, thresholds, classes, fallback_class=0):
    """Apply per-class thresholds; samples passing no class fall back to fallback_class."""
    classes = np.array(classes)
    thr_arr = np.array([thresholds[c] for c in classes])
    passed = proba >= thr_arr[None, :]
    masked = np.where(passed, proba, -np.inf)
    preds = classes[masked.argmax(1)]
    preds[~passed.any(1)] = fallback_class
    return preds

def class_weight_dict(y, cap=CLASS_WEIGHT_CAP):
    """Balanced class weights as {class_label: weight}, capped so the ratio between the largest
    and smallest weight never exceeds `cap` (any weight above cap * min_weight is clipped down to
    that ceiling). This is the single source of truth for imbalance weighting - LightGBM, XGBoost,
    RandomForest (all via sample_weight), MLP (via oversampling below), and both GPU variants all
    derive their weighting from this same capped dict, so every model uses the identical ratio.
    Returns None if class weighting is off."""
    if not USE_CLASS_WEIGHT:
        return None
    classes = np.unique(y)
    raw_weights = compute_class_weight("balanced", classes=classes, y=y)
    min_w = raw_weights.min()
    capped_weights = np.minimum(raw_weights, min_w * cap)
    return dict(zip(classes, capped_weights))

def class_weights(y, cap=CLASS_WEIGHT_CAP):
    """Per-sample weights expanded from the capped class_weight_dict() above - used as
    `sample_weight` for LightGBM, XGBoost, and RandomForest (all trained via sample_weight rather
    than a "balanced"/"balanced_subsample" string, so the same cap applies to all three)."""
    cw = class_weight_dict(y, cap)
    if cw is None:
        return None
    y_arr = np.asarray(y)
    return np.array([cw[v] for v in y_arr])

def preprocess_ml(X_tr, y_tr, X_va, num=None, high=None, low=None):
    num = list(numeric_cols) if num is None else list(num)
    high = list(high_card) if high is None else list(high)
    low = list(low_card) if low is None else list(low)
    X_tr, X_va = X_tr.copy(), X_va.copy()
    original_columns = list(X_tr.columns)
    freq_maps = {}
    for c in high:
        f = X_tr[c].value_counts(normalize=True)
        freq_maps[c] = f
        X_tr[c+"_freq"] = X_tr[c].map(f).fillna(0.0).astype(np.float32)
        X_va[c+"_freq"] = X_va[c].map(f).fillna(0.0).astype(np.float32)
        num.append(c+"_freq")
    X_tr, X_va = X_tr.drop(columns=high), X_va.drop(columns=high)
    try:
        from sklearn.preprocessing import TargetEncoder
        enc = TargetEncoder(random_state=RANDOM_STATE)
    except Exception:
        from sklearn.preprocessing import OneHotEncoder
        enc = OneHotEncoder(handle_unknown="ignore", min_frequency=5)
    pre = ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
        ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")), ("e", enc)]), low),
    ])
    X_tr_p = pre.fit_transform(X_tr, y_tr)
    pre.high_card_freq_maps_ = freq_maps
    pre.high_card_columns_ = high
    pre.input_columns_ = original_columns
    return X_tr_p, pre.transform(X_va), pre


def dense32(a):
    return np.asarray(a.toarray() if sp.issparse(a) else a, dtype=np.float32)

def save_scores(name, model, Xval, Xtest, yval, ytest, out_dir, pre_labels, threshold_store,
                tuned_scores, argmax_scores, predictions):
    val_proba = np.asarray(model.predict_proba(Xval))
    test_proba = np.asarray(model.predict_proba(Xtest))
    classes = list(range(n_classes))
    thresholds = tune_thresholds(yval, val_proba, classes)
    majority = int(pd.Series(y_fit).mode()[0])
    tuned = predict_with_thresholds(test_proba, thresholds, classes, majority)
    argmax = test_proba.argmax(axis=1)
    threshold_store[name] = thresholds
    tuned_scores[name] = calc_metrics(ytest, tuned)
    argmax_scores[name] = calc_metrics(ytest, argmax)
    predictions[name] = np.asarray(pre_labels)[tuned]
    cm = confusion_matrix(ytest, tuned, labels=classes)
    pd.DataFrame(cm, index=pre_labels, columns=pre_labels).to_csv(out_dir / f'{name}_confusion_matrix.csv')
    pd.DataFrame(classification_report(ytest, tuned, labels=classes,
        target_names=[str(c) for c in pre_labels], output_dict=True, zero_division=0)).T.to_csv(
        out_dir / f'{name}_classification_report.csv')
    print(name, 'threshold-tuned TEST:', tuned_scores[name])

## Five-fold CV on the train file

In [ ]:
def make_models():
    return {
        'LightGBM': lgb.LGBMClassifier(n_estimators=250, learning_rate=0.05,
            num_leaves=31, n_jobs=-1, verbosity=-1, device_type=LGB_DEVICE, random_state=RANDOM_STATE),
        'XGBoost': xgb.XGBClassifier(n_estimators=250, max_depth=8, learning_rate=0.05,
            tree_method='hist', device='cuda', objective='multi:softprob', num_class=n_classes,
            eval_metric='mlogloss', n_jobs=-1, random_state=RANDOM_STATE),
        'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=15,
            min_samples_leaf=20, max_features=0.3, n_jobs=-1, random_state=RANDOM_STATE)
    }

cv_rows = []
skf = StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=RANDOM_STATE)
for fold,(tr,va) in enumerate(skf.split(X_dev,y_dev),1):
    Xtr,Xva = X_dev.iloc[tr],X_dev.iloc[va]
    ytr,yva = y_dev[tr],y_dev[va]
    Xtr_p,Xva_p,_ = preprocess_ml(Xtr,ytr,Xva)
    sw = class_weights(ytr)
    for name,model in make_models().items():
        model.fit(Xtr_p,ytr,sample_weight=sw)
        scores=calc_metrics(yva,model.predict(Xva_p))
        cv_rows.append({'Model':name,'Fold':fold,**scores})
        print(name,'fold',fold,scores)
cv_df=pd.DataFrame(cv_rows)
cv_df.to_csv(OUT_DIR/'cv_fold_metrics.csv',index=False)
cv_df.groupby('Model')[METRICS].agg(['mean','std']).to_csv(OUT_DIR/'validation_summary.csv')

## Final models, threshold tuning, and the held-out test

In [ ]:
Xtr_p, Xrest_p, pre = preprocess_ml(X_fit,y_fit,pd.concat([X_val,X_test]))
Xval_p, Xtest_p = Xrest_p[:len(X_val)],Xrest_p[len(X_val):]
joblib.dump(pre, OUT_DIR/'preprocess.pkl')
weights=class_weights(y_fit)
thresholds, tuned_scores, argmax_scores = {}, {}, {}
predictions=pd.DataFrame({'y_true':np.asarray(classes)[y_test]})
for name,model in make_models().items():
    model.fit(Xtr_p,y_fit,sample_weight=weights)
    joblib.dump(model,OUT_DIR/f'{name}.pkl')
    save_scores(name,model,Xval_p,Xtest_p,y_val,y_test,OUT_DIR,classes,
                thresholds,tuned_scores,argmax_scores,predictions)
joblib.dump(thresholds,OUT_DIR/'class_thresholds.pkl')
pd.DataFrame(tuned_scores).T[METRICS].to_csv(OUT_DIR/'test_comparison_matrix.csv')
pd.DataFrame(argmax_scores).T[METRICS].to_csv(OUT_DIR/'test_comparison_matrix_argmax_only.csv')
predictions.to_csv(OUT_DIR/'test_predictions.csv',index=False)
print(pd.DataFrame(tuned_scores).T[METRICS].sort_values('Macro_F1',ascending=False))
print('Saved to:',OUT_DIR.resolve())